In [ ]:
import subprocess, sys, traceback, os

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"$ {' '.join(cmd)}\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}", flush=True)
    r.check_returncode()
    return r

try:
    ckpt("STEP0: /kaggle/input listing")
    for root, dirs, files in os.walk("/kaggle/input"):
        ckpt(f"  {root}: dirs={dirs} files={files[:10]}")

    ckpt("STEP1: pip install tabicl")
    run([sys.executable, "-m", "pip", "install", "-q", "tabicl"])

    ckpt("STEP1: git clone")
    subprocess.run(["rm", "-rf", "kaggle_playground_s6e9"])
    run(["git", "clone", "-q", "https://github.com/gccarno/kaggle_playground_s6e9.git"])

    import torch
    ckpt(f"STEP1: cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        ckpt(f"STEP1: device: {torch.cuda.get_device_name(0)}")
        ckpt(f"STEP1: mem GB: {torch.cuda.get_device_properties(0).total_memory / 1e9}")
    ckpt("STEP1: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP1 FAILED\n" + tb)
    raise

In [ ]:
import json, os, subprocess, sys, traceback

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

# Every kernel-based probe edits ONLY this cfg dict before pushing -- the notebook IS
# the config, the same way run_local.py's --cfg is for local runs (CLAUDE.md).
cfg = {
    "run_tag": "P0_smoke",
    "learner": "tabicl",
    "n_folds": 2,
    "te_cols": ["Annual_Income_USD", "Daily_Commute_km", "Age"],
    "te_smooth": 5.0,
    "te_backoff": "neighborhood",
    "tabicl_max_context": 3000,
    "tabicl_predict_chunk": 20000,
    "tabicl_n_estimators": 4
}

try:
    ckpt("STEP2: running pipeline.py")
    env = dict(os.environ, S6E9_CFG=json.dumps(cfg))
    r = subprocess.run([sys.executable, "-u", "kaggle_playground_s6e9/src/pipeline.py"],
                       env=env, capture_output=True, text=True)
    print(r.stdout, flush=True)
    print(r.stderr, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP2 stdout tail:\n" + r.stdout[-4000:] + "\nSTEP2 stderr tail:\n" + r.stderr[-4000:])
    r.check_returncode()
    ckpt("STEP2: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP2 FAILED\n" + tb)
    raise